In [0]:
# =============================================================
# NOTEBOOK:  05_bronze_jdbc_ingestion
# PURPOSE:   Ingest ERP tables from Azure SQL into Bronze Delta
# SOURCE:    Azure SQL Database (retailmax-erp) via JDBC
# TARGET:    Bronze Delta Tables (bronze/tables/erp_sql/)
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid
from delta.tables import DeltaTable



# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ── Storage Paths ─────────────────────────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

# ── Retrieve Secrets ──────────────────────────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

# ── Configure Spark for ADLS ──────────────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

# ── Retrieve Secrets ──────────────────────────────────────────
sql_server     = dbutils.secrets.get(scope=scope_name, key="sql-server")
sql_database = dbutils.secrets.get(scope=scope_name, key="sql-database")
sql_user     = dbutils.secrets.get(scope=scope_name, key="sql-user")
sql_password     = dbutils.secrets.get(scope=scope_name, key="sql-password")

jdbc_url = f'jdbc:sqlserver://{sql_server}:1433;database={sql_database};encrypt=true;trustServerCertificate=false; hostNameInCertificate=*.database.windows.net;loginTimeout=30;'

jdbc_properties = {
    "user": sql_user,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"  
}



In [0]:
def ingest_sql_table(source_table, target_path, source_system="erp_sql", load_type='Full', watermark_column=None,control_table_path=None):

    print("=" * 50)
    print(f"Source : {source_table}")
    print(f"Target : {target_path}")
    print(f"Mode   : {load_type}")
    print("=" * 50)
    batch_id = uuid.uuid4().hex
    source_count = 0
    target_count = 0

    if load_type == 'Full':

        df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", source_table) \
        .options(**jdbc_properties)\
        .load()

        df = (df
            .withColumn("_source_system", F.lit(source_system))
            .withColumn("_source_table", F.lit(source_table))
            .withColumn("_load_timestamp", F.current_timestamp())
            .withColumn("_load_date", F.current_date())
            .withColumn("_batch_id", F.lit(batch_id))
        )

        df.write\
            .mode('overwrite')\
            .format('delta')\
            .save(target_path)

        # Source count
        df_source = spark.read.format("jdbc")\
            .option("url", jdbc_url)\
            .option("dbtable", f"(SELECT COUNT(*) AS cnt FROM {source_table}) AS t")\
            .options(**jdbc_properties)\
            .load()

        # Target count
        df_target = spark.read.format("delta")\
            .load(target_path)

        source_count = df_source.collect()[0]['cnt']
        target_count = df_target.count()
    
    else:

        df_control = (
            spark.read.format("delta").load(control_table_path)
            .filter(
                (F.col("source_system") == source_system) &
                (F.col("source_table") == source_table)
            )
        )

        last_watermark = df_control.select("last_watermark").first()["last_watermark"]
        if last_watermark is None:
            raise ValueError(f"No watermark found for {source_table} in control table")
        last_watermark = last_watermark.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        print(f"Last Watermark: {last_watermark}")

        query = f"(SELECT * FROM {source_table} WHERE {watermark_column} > '{last_watermark}') AS src"

        df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", query) \
        .options(**jdbc_properties)\
        .load()

        if df.count() == 0:
            print("No new records found.")

        else:
        
            df = (df
                .withColumn("_source_system", F.lit(source_system))
                .withColumn("_source_table", F.lit(source_table))
                .withColumn("_load_timestamp", F.current_timestamp())
                .withColumn("_load_date", F.current_date())
                .withColumn("_batch_id", F.lit(batch_id))
            )

            df.write\
                .mode('append')\
                .format('delta')\
                .save(target_path)

            latest_watermark = (
                df
                .agg(F.max(F.col(watermark_column)).alias("latest_watermark"))
                .collect()[0]["latest_watermark"]
            )
            print(f"Latest Watermark: {latest_watermark}")

            control_table = DeltaTable.forPath(spark, control_table_path)

            control_table.update(
                condition=f"source_system = '{source_system}' AND source_table = '{source_table}'",
                set={
                    "last_watermark": F.lit(latest_watermark),
                    "last_run_timestamp": F.current_timestamp(),
                    "last_batch_id": F.lit(batch_id)
                }
            )

            #Source count
            source_count = df.count()

        # Target count
        df_target = spark.read.format("delta")\
            .load(target_path)\
            .filter(F.col('_batch_id')==batch_id)

        target_count = df_target.count()        

    print(f"Batch ID      : {batch_id}")
    print(f"Source count  : {source_count}")
    print(f"Target count  : {target_count}")

    if source_count == target_count:
        print("✅ Validation passed")
    else:
        print("⚠️  WARNING: count mismatch!")

    print("=" * 50)


In [0]:
# dim_stores
ingest_sql_table(
    source_table="dbo.dim_stores",
    target_path=bronze_path + "tables/erp_sql/dim_stores/",
    source_system="erp_sql"
)

# dim_products
ingest_sql_table(
    source_table="dbo.dim_products",
    target_path=bronze_path + "tables/erp_sql/dim_products/",
    source_system="erp_sql"
)

# erp_sales_orders
ingest_sql_table(
    source_table="dbo.erp_sales_orders",
    target_path=bronze_path + "tables/erp_sql/erp_sales_orders/",
    source_system="erp_sql",
    load_type='Incremental',
    control_table_path=bronze_path + "metadata/control_tables/jdbc_watermarks/",
    watermark_column="updated_at"
)

In [0]:
# ── Final Verification ────────────────────────────────────────
tables = {
    "dim_stores":        bronze_path + "tables/erp_sql/dim_stores/",
    "dim_products":      bronze_path + "tables/erp_sql/dim_products/",
    "erp_sales_orders":  bronze_path + "tables/erp_sql/erp_sales_orders/"
}

print("=" * 50)
print("Bronze Layer - JDBC Ingestion Summary")
print("=" * 50)
for table_name, path in tables.items():
    count = spark.read.format("delta").load(path).count()
    print(f"{table_name:<25} {count:>6} rows")
print("=" * 50)

In [0]:
df_max = spark.read.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "(SELECT MAX(updated_at) AS max_wm FROM dbo.erp_sales_orders) AS t")\
    .options(**jdbc_properties)\
    .load()

max_watermark = df_max.collect()[0]["max_wm"]
print(f"Max watermark: {max_watermark}")

In [0]:
from datetime import datetime

control_table_path = bronze_path + "metadata/control_tables/jdbc_watermarks/"

# # Get batch_id from Bronze sales orders table
# df_batch = (
#     spark.read.format("delta")
#     .load(bronze_path + "tables/erp_sql/erp_sales_orders/")
#     .select("_batch_id")
#     .distinct()
# )

# last_batch_id = df_batch.collect()[0]["_batch_id"]

# max_watermark already fetched from Azure SQL
# Example:
# max_watermark = df_max.collect()[0]["max_wm"]

watermark_data = [
    {
        "source_system": "erp_sql",
        "source_table": "dbo.erp_sales_orders",
        "watermark_column": "updated_at",
        "last_watermark": "1900-01-01 00:00:00.000000",
        "last_run_timestamp": datetime.now(),
        "last_batch_id": last_batch_id
    }
]

df_watermark = spark.createDataFrame(watermark_data)

df_watermark = df_watermark.withColumn(
    "last_watermark",
    F.to_timestamp("last_watermark", "yyyy-MM-dd HH:mm:ss.SSSSSS")
)

df_watermark.show(truncate=False)
df_watermark.printSchema()

df_watermark.write \
    .format("delta") \
    .mode("overwrite") \
    .save(control_table_path)